# Diagnosing Training Health with Gradient and Activation Statistics

A training run is a black box by default. Loss goes down — or it doesn't — and you have almost no visibility into why. This notebook gives you the instrumentation to open that box.

The central diagnostic is the **gradient-to-weight ratio**: $\rho_l = \|\nabla W_l\| / \|W_l\|$ for each layer $l$. This single number tells you whether each layer is learning at a healthy rate, stuck, or about to explode. But to use it well you need to understand what controls it — and there are exactly five levers: initialization, normalization, learning rate and schedule, gradient clipping, and residual stream scaling. Breaking each lever one at a time on a live training run and watching what happens to the chart teaches you to recognize failure modes on sight.

Once you can read gradient diagnostics, the next question is where to display them. We build a `TrainingLogger` that writes every metric to a JSONL file, a `LiveDashboard` that updates in real time, and a TensorBoard integration for post-hoc run comparison. Finally, we collect all eight common training pathologies — plateau, spike, divergence, vanishing gradients, exploding gradients, NaN propagation, dead neurons, overfitting — with minimal reproducible examples for each and a `HealthReport` class that detects them automatically.

## The Hook API

PyTorch hooks are callbacks that fire at specific points in the forward or backward pass. They observe the computation without modifying it.

`module.register_forward_hook(fn)` fires immediately after `module.forward()` returns, with signature `fn(module, input, output) -> None`. `input` is a tuple of the module's inputs; `output` is the module's output tensor. This is where you capture activations:

In [ ]:
import torch
import torch.nn as nn

layer = nn.Linear(64, 64)
activations = {}

def forward_hook(module, input, output):
    activations['linear'] = output.detach()  # <1>

handle = layer.register_forward_hook(forward_hook)

x = torch.randn(8, 64)
_ = layer(x)
print(activations['linear'].shape)   # torch.Size([8, 64])

handle.remove()  # <2>

1. Always call `.detach()` inside hooks. Without it you keep a reference to the computation graph node, preventing it from being freed after `.backward()` — a memory leak that grows every step.
2. Store every handle and call `.remove()` when done. Hooks are stored in the module and fire on every forward pass. Accumulating duplicate hooks slows training.

`module.register_full_backward_hook(fn)` fires during the backward pass with signature `fn(module, grad_input, grad_output) -> None`. `grad_output[0]` is the gradient flowing *into* this module from downstream — this is what you want for diagnosing gradient flow:

In [ ]:
gradients = {}

def backward_hook(module, grad_input, grad_output):
    if grad_output[0] is not None:
        gradients['linear'] = grad_output[0].detach()

handle = layer.register_full_backward_hook(backward_hook)

x = torch.randn(8, 64, requires_grad=True)
out = layer(x)
out.sum().backward()

print(gradients['linear'].shape)   # torch.Size([8, 64])
handle.remove()

For per-parameter gradient inspection, register a hook directly on the parameter tensor. This fires after gradient accumulation but before the optimizer reads it — a clean view of the raw gradient:

In [ ]:
layer = nn.Linear(64, 64)
param_grads = {}

def param_grad_hook(grad):
    param_grads['weight'] = grad.detach().clone()
    return grad   # must return grad (or None to zero it)

handle = layer.weight.register_hook(param_grad_hook)

x = torch.randn(8, 64)
layer(x).sum().backward()
print(param_grads['weight'].norm().item())
handle.remove()

## The `GradientMonitor` Class

We build a `GradientMonitor` that attaches to every named `Linear` module in a model, records statistics every step, and computes the gradient-to-weight ratio. The `LayerStats` dataclass holds one record per layer per step:

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass, field
import math


@dataclass
class LayerStats:
    """Statistics for one layer at one training step."""
    step:              int
    layer_name:        str
    act_mean:          float = 0.0
    act_std:           float = 0.0
    act_frac_zero:     float = 0.0   # fraction of zero activations (dead neurons)
    grad_mean:         float = 0.0
    grad_std:          float = 0.0
    grad_l2:           float = 0.0
    grad_max:          float = 0.0
    weight_l2:         float = 0.0
    grad_weight_ratio: float = 0.0   # grad_l2 / weight_l2


class GradientMonitor:
    """
    Attaches forward and backward hooks to every named Linear module.
    Records per-layer activation and gradient statistics every step.

    Usage:
        monitor = GradientMonitor(model)
        monitor.attach()
        for step, batch in enumerate(dataloader):
            loss = train_step(batch)
            monitor.record(step)         # call after loss.backward()
        monitor.detach()
        df = monitor.to_dataframe()
    """

    def __init__(self, model: nn.Module):
        self.model   = model
        self.history: list[LayerStats] = []
        self._handles = []
        self._act_buffer:  dict[str, torch.Tensor] = {}
        self._grad_buffer: dict[str, torch.Tensor] = {}

    def attach(self):
        """Register hooks on all named Linear modules."""
        for name, module in self.model.named_modules():
            if not isinstance(module, nn.Linear):
                continue

            def make_fwd(n):
                def hook(mod, inp, out):
                    self._act_buffer[n] = out.detach()
                return hook

            def make_bwd(n):
                def hook(mod, g_in, g_out):
                    if g_out[0] is not None:
                        self._grad_buffer[n] = g_out[0].detach()
                return hook

            self._handles.append(module.register_forward_hook(make_fwd(name)))
            self._handles.append(module.register_full_backward_hook(make_bwd(name)))

    def detach(self):
        """Remove all hooks."""
        for h in self._handles:
            h.remove()
        self._handles.clear()

    def record(self, step: int):
        """Compute LayerStats for every monitored layer and append to history."""
        for name, module in self.model.named_modules():
            if not isinstance(module, nn.Linear):
                continue

            act  = self._act_buffer.get(name)
            grad = self._grad_buffer.get(name)

            w_l2   = module.weight.norm().item()
            g_l2   = module.weight.grad.norm().item() if module.weight.grad is not None else 0.0
            ratio  = g_l2 / (w_l2 + 1e-8)

            stats = LayerStats(
                step=step,
                layer_name=name,
                act_mean=act.mean().item()              if act is not None else 0.0,
                act_std=act.std().item()                if act is not None else 0.0,
                act_frac_zero=(act.abs() < 0.01).float().mean().item()
                                                        if act is not None else 0.0,
                grad_mean=grad.mean().item()            if grad is not None else 0.0,
                grad_std=grad.std().item()              if grad is not None else 0.0,
                grad_l2=g_l2,
                grad_max=grad.abs().max().item()        if grad is not None else 0.0,
                weight_l2=w_l2,
                grad_weight_ratio=ratio,
            )
            self.history.append(stats)

        self._act_buffer.clear()
        self._grad_buffer.clear()

    def warn_if_unhealthy(self, lo=1e-3, hi=1e-2):
        """Print a warning for any layer whose ratio is outside [lo, hi]."""
        recent = {s.layer_name: s for s in self.history[-100:]}
        for name, s in recent.items():
            if s.grad_weight_ratio < lo:
                print(f"  WARN {name}: ρ={s.grad_weight_ratio:.2e} < {lo:.0e} (vanishing?)")
            elif s.grad_weight_ratio > hi:
                print(f"  WARN {name}: ρ={s.grad_weight_ratio:.2e} > {hi:.0e} (exploding?)")

    def to_dataframe(self):
        import pandas as pd
        return pd.DataFrame([vars(s) for s in self.history])

## The Gradient-to-Weight Ratio: What Healthy Looks Like

The healthy range for $\rho_l = \|\nabla W_l\| / \|W_l\|$ is roughly $[10^{-3}, 10^{-2}]$. This means each update moves the weights by about 0.1% to 1% of their current magnitude per step.

We train the nano GPT from [NB01](/courses/llm/01-gpt-architecture.html) for 200 steps with correct settings and record the ratio. The baseline chart — uniformly green across all layers and all steps — is the reference we compare every broken run against:

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from notebook_01 import GPT, NanoGPTConfig   # implementations from NB01
from notebook_02 import Tokenizer             # BPE tokenizer from NB02

config = NanoGPTConfig()
model  = GPT(config)
tok    = Tokenizer.load('nano_tokenizer.json')

text   = open('tinyshakespeare.txt').read()
data   = torch.tensor(tok.encode(text), dtype=torch.long)
block  = config.max_seq_len

def get_batch(batch_size=8):
    ix = torch.randint(len(data) - block, (batch_size,))
    x  = torch.stack([data[i   : i+block  ] for i in ix])
    y  = torch.stack([data[i+1 : i+block+1] for i in ix])
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
monitor   = GradientMonitor(model)
monitor.attach()

losses = []
for step in range(200):
    x, y = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    monitor.record(step)
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"step {step:3d}  loss={loss.item():.4f}")
        monitor.warn_if_unhealthy()

monitor.detach()
df = monitor.to_dataframe()

We plot the ratio as a heatmap (x=step, y=layer, color=log₁₀(ratio)). The color scale goes from blue (vanishing, $< 10^{-5}$) through green (healthy, $10^{-3}$ to $10^{-2}$) to red (exploding, $> 10^{-1}$):

In [ ]:
#| code-fold: true
def plot_ratio_heatmap(df, title="Gradient-to-Weight Ratio"):
    layers = df['layer_name'].unique()
    steps  = df['step'].unique()
    matrix = np.full((len(layers), len(steps)), np.nan)
    layer_idx = {l: i for i, l in enumerate(layers)}
    step_idx  = {s: i for i, s in enumerate(steps)}
    for _, row in df.iterrows():
        i = layer_idx[row['layer_name']]
        j = step_idx[row['step']]
        matrix[i, j] = math.log10(row['grad_weight_ratio'] + 1e-10)
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(
        matrix, aspect='auto', cmap='RdYlGn',
        vmin=-5, vmax=-1, origin='upper',
    )
    ax.set_yticks(range(len(layers)))
    ax.set_yticklabels(layers, fontsize=8)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Layer')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='log₁₀(‖∇W‖ / ‖W‖)')
    plt.tight_layout()
    plt.show()

plot_ratio_heatmap(df, title="Baseline — Healthy Training")

## The Five Levers

We now break each lever one at a time and watch what happens to the chart. Each experiment runs the same 200 steps on the same data with one setting changed.

### Lever 1: Initialization

**What it controls:** The starting value of $\|W\|$ — the denominator of $\rho$. If weights are initialized too small, $\|W\|$ is small and $\rho$ appears healthy even though weights barely move in absolute terms. If weights are initialized too large, activations saturate immediately and gradients through the saturating nonlinearity vanish:

In [ ]:
def run_with_init(std: float, label: str, steps: int = 200):
    model = GPT(config)
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)
    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label)

df_init_ok   = run_with_init(std=0.02,  label='init std=0.02 (correct)')
df_init_tiny = run_with_init(std=0.001, label='init std=0.001 (too small)')
df_init_huge = run_with_init(std=0.5,   label='init std=0.5 (too large)')

- `std=0.02`: Green heatmap, ratio stable around $10^{-2.5}$.
- `std=0.001`: Ratio appears artificially high — tiny $\|W\|$ inflates $\rho$ — but weights barely move in absolute terms.
- `std=0.5`: Ratio is tiny — large $\|W\|$ deflates $\rho$; the softmax inside attention saturates immediately, producing near-zero gradients.

[The lesson: initialization sets the scale of the system. Both numerator and denominator of $\rho$ depend on it, so the ratio alone is not sufficient — always plot absolute gradient norms alongside the ratio.]{.underline}

### Lever 2: Normalization (RMSNorm)

**What it controls:** Whether activation magnitudes are bounded across layers. Without norm, activations grow or shrink as they propagate through layers — gradients follow. With pre-norm, each layer sees normalized inputs regardless of what earlier layers did:

In [ ]:
def run_with_model(model_factory, label, steps=200):
    model     = model_factory()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        if torch.isnan(loss):
            print(f"  NaN at step {step} — stopping")
            break
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)
    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label)

df_with_norm = run_with_model(lambda: GPT(config),          label='With RMSNorm')
df_no_norm   = run_with_model(lambda: GPT(config, norm=False), label='No norm')

- With norm: ratio is uniform across layers — each block sees the same activation scale.
- Without norm: ratio degrades with depth — activation scale compounds across layers; gradients flow unevenly; the run often produces NaN within 100 steps.

[This is the quantitative reason pre-norm matters for deep models.]{.mark} BN normalizes across the batch (wrong dimension); LN/RMSNorm normalize across features (correct dimension for language models).

### Lever 3: Learning Rate

**What it controls:** $\|\Delta W\| = \text{lr} \cdot \|\nabla W\|$. The ratio $\rho = \|\nabla W\| / \|W\|$ does not include the LR — but the *effective* update ratio $\|\Delta W\| / \|W\| = \text{lr} \cdot \rho$ does:

In [ ]:
def run_with_lr(lr: float, label: str, steps: int = 200):
    model     = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    monitor   = GradientMonitor(model)
    monitor.attach()
    losses = []
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        if torch.isnan(loss):
            break
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)
        losses.append(loss.item())
    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label), losses

df_lr_good, losses_good = run_with_lr(3e-4,  label='lr=3e-4 (good)')
df_lr_low,  losses_low  = run_with_lr(1e-6,  label='lr=1e-6 (too low)')
df_lr_high, losses_high = run_with_lr(3e-2,  label='lr=3e-2 (too high)')

The raw gradient-to-weight ratio $\rho$ looks similar for all three LRs — it is independent of LR. [The gradient-to-weight ratio is not sufficient on its own.]{.mark} You must plot it alongside the loss curve and the effective update ratio $\text{lr} \cdot \rho$. The dashboard in the next section shows all three.

### Lever 4: Gradient Clipping

**What it controls:** A hard ceiling on the global gradient norm $\|g\| = \sqrt{\sum_l \|\nabla W_l\|^2}$. When $\|g\| > c$, all gradients are rescaled by $c / \|g\|$, preventing individual large-gradient events from destabilizing training:

In [ ]:
def run_with_clip(clip: float, label: str, steps: int = 200):
    model     = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()
    grad_norms = []
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        raw_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip).item()
        grad_norms.append(raw_norm)
        optimizer.step()
        monitor.record(step)
    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label), grad_norms

df_clip1,  norms_clip1  = run_with_clip(1.0, 'clip=1.0 (standard)')
df_clip01, norms_clip01 = run_with_clip(0.1, 'clip=0.1 (aggressive)')
df_noclip, norms_noclip = run_with_clip(1e9, 'no clipping')

The pre-clip gradient norm is the key signal. If it is consistently near the clip threshold, the model is perpetually constrained — either reduce LR or increase clip. If it never approaches the threshold, clipping is irrelevant and can be raised. If it spikes occasionally to 10× the threshold, clipping is doing its job:

In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(norms_clip1,  label='raw norm (clip=1.0)', alpha=0.8)
ax.semilogy(norms_noclip, label='raw norm (no clip)',  alpha=0.6)
ax.axhline(1.0, color='red', linestyle='--', label='clip threshold')
ax.set_xlabel('Step')
ax.set_ylabel('Global Gradient Norm')
ax.set_title('Pre-Clip Gradient Norm — clip=1.0 vs no clipping')
ax.legend()
plt.tight_layout()
plt.show()

### Lever 5: Residual Stream Scaling

**What it controls:** How much each residual block contributes to the stream at initialization. Without the $1/\sqrt{2L}$ scaling on output projections (see [NB01](/courses/llm/01-gpt-architecture.html)), the stream variance grows as $O(L)$ — for 12 layers it is 12× larger than intended, causing gradient norms to vary systematically with depth. Deeper layers contribute more and receive larger gradients; early layers barely receive gradient signal in the first steps:

In [ ]:
def make_model_no_resid_scaling():
    """Reset residual output projections to standard init (no 1/√(2L) scaling)."""
    model = GPT(config)
    n_layers = config.n_layers
    for name, module in model.named_modules():
        if hasattr(module, '_is_residual_proj') and module._is_residual_proj:
            nn.init.normal_(module.weight, 0.0, 0.02)   # revert to standard
    return model

def run_model(model, label, steps=200):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    monitor   = GradientMonitor(model)
    monitor.attach()
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        monitor.record(step)
    monitor.detach()
    return monitor.to_dataframe().assign(experiment=label)

df_scaled   = run_model(GPT(config),              'With 1/√(2L) scaling')
df_unscaled = run_model(make_model_no_resid_scaling(), 'Without scaling')

- With scaling: per-layer ratios are roughly uniform across depth — each layer contributes a similar magnitude update.
- Without scaling: ratio degrades monotonically with depth — final layers train much faster than early layers.

For a 6-layer nano model the effect is moderate. [At 96 layers (GPT-3 scale) without this scaling, the first 20 layers effectively do not train for hundreds of steps — a catastrophic waste of compute.]{.underline}

### The Combined Chart

All five levers on one figure — each as a subplot showing the mean ratio across all layers over time:

In [ ]:
#| code-fold: true
import pandas as pd

def mean_ratio_per_step(df_exp):
    return df_exp.groupby('step')['grad_weight_ratio'].mean()

experiments = {
    'Baseline (all correct)':     df,
    'Init std=0.001 (too small)': df_init_tiny,
    'Init std=0.5 (too large)':   df_init_huge,
    'No normalization':           df_no_norm,
    'No residual scaling':        df_unscaled,
}

fig, axes = plt.subplots(len(experiments), 1, figsize=(12, 14), sharex=True)
for ax, (label, df_exp) in zip(axes, experiments.items()):
    ratio = mean_ratio_per_step(df_exp)
    ax.semilogy(ratio.index, ratio.values, linewidth=2)
    ax.axhline(1e-3, color='green', linestyle='--', alpha=0.5, label='healthy lo')
    ax.axhline(1e-2, color='green', linestyle='--', alpha=0.5, label='healthy hi')
    ax.set_ylabel('mean ρ')
    ax.set_title(label)
    ax.set_ylim(1e-6, 1e0)
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Training Step')
fig.suptitle('Five Levers on the Gradient-to-Weight Ratio', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Dead Neurons via Activation Hooks

A **dead neuron** is a unit that outputs near-zero for every input in a batch — its weights have been pushed into the permanently-negative region and it no longer participates in computation. The gradient through a dead ReLU is exactly zero, so the neuron cannot recover via gradient descent.

We detect this via the `act_frac_zero` field in `LayerStats` — the fraction of activation values near zero in the forward pass. SwiGLU (our FFN activation) does not produce exactly-zero outputs, so we use the `abs(act) < 0.01` proxy:

In [ ]:
#| code-fold: true
def plot_dead_neurons(df, threshold=0.5):
    fig, ax = plt.subplots(figsize=(12, 5))
    for layer_name, group in df.groupby('layer_name'):
        if 'ffn' not in layer_name:
            continue
        ax.plot(group['step'], group['act_frac_zero'],
                label=layer_name, alpha=0.7)
    ax.axhline(threshold, color='red', linestyle='--',
               label=f'dead threshold ({threshold})')
    ax.set_xlabel('Step')
    ax.set_ylabel('Fraction near-zero Activations')
    ax.set_title('Dead Neuron Monitor — FFN Layers')
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

plot_dead_neurons(df)

## The `TrainingLogger` — Structured Logging First

Before any visualization, write everything to disk in a structured format. [Visualization tools come and go; a JSONL file is forever.]{.mark}

The `StepRecord` dataclass holds one record per step. The logger appends each record as a JSON line to `metrics.jsonl` — append-only and crash-safe. If the job dies at step 8,432 you have records for steps 0–8,431 and can resume:

In [ ]:
import json
import time
import os
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional


@dataclass
class StepRecord:
    """One record per training step."""
    step:             int
    timestamp:        float
    train_loss:       float
    learning_rate:    float
    global_grad_norm: float
    tokens_per_sec:   float
    eval_loss:        Optional[float] = None
    mean_grad_ratio:  Optional[float] = None
    min_grad_ratio:   Optional[float] = None
    max_grad_ratio:   Optional[float] = None
    mean_weight_norm: Optional[float] = None
    gpu_memory_gb:    Optional[float] = None


class TrainingLogger:
    """
    Writes training metrics to a JSONL file (one JSON object per line).
    Optionally feeds TensorBoard.

    Usage:
        logger = TrainingLogger(run_dir='runs/my_experiment')
        for step in range(max_steps):
            logger.start_step(total_tokens)
            ...forward/backward...
            logger.log_step(step, train_loss, lr, grad_norm, tps)
    """

    def __init__(
        self,
        run_dir: str,
        run_name: str = None,
        use_tensorboard: bool = False,
    ):
        self.run_dir  = Path(run_dir)
        self.run_name = run_name or f"run_{int(time.time())}"
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.log_path  = self.run_dir / 'metrics.jsonl'
        self._log_file = open(self.log_path, 'a')
        self._step_start_time   = time.time()
        self._step_start_tokens = 0
        self._records: list[StepRecord] = []
        self._writer = None
        if use_tensorboard:
            try:
                from torch.utils.tensorboard import SummaryWriter
                self._writer = SummaryWriter(log_dir=str(self.run_dir / 'tb'))
            except ImportError:
                print("TensorBoard not installed — logging to JSONL only")

    def start_step(self, total_tokens: int):
        self._step_start_time   = time.time()
        self._step_start_tokens = total_tokens

    def log_step(
        self,
        step: int,
        train_loss: float,
        learning_rate: float,
        global_grad_norm: float,
        total_tokens: int,
        eval_loss: float = None,
        mean_grad_ratio: float = None,
        min_grad_ratio:  float = None,
        max_grad_ratio:  float = None,
        mean_weight_norm: float = None,
    ):
        elapsed = time.time() - self._step_start_time + 1e-9
        tps = (total_tokens - self._step_start_tokens) / elapsed
        gpu_mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

        record = StepRecord(
            step=step,
            timestamp=time.time(),
            train_loss=train_loss,
            learning_rate=learning_rate,
            global_grad_norm=global_grad_norm,
            tokens_per_sec=tps,
            eval_loss=eval_loss,
            mean_grad_ratio=mean_grad_ratio,
            min_grad_ratio=min_grad_ratio,
            max_grad_ratio=max_grad_ratio,
            mean_weight_norm=mean_weight_norm,
            gpu_memory_gb=gpu_mem,
        )
        self._records.append(record)
        self._log_file.write(json.dumps(asdict(record)) + '\n')
        self._log_file.flush()

        if self._writer:
            self._writer.add_scalar('loss/train',    train_loss,       step)
            self._writer.add_scalar('grad/norm',     global_grad_norm, step)
            self._writer.add_scalar('lr',            learning_rate,    step)
            self._writer.add_scalar('throughput/tps', tps,             step)
            if eval_loss is not None:
                self._writer.add_scalar('loss/eval', eval_loss, step)

    def close(self):
        self._log_file.close()
        if self._writer:
            self._writer.close()

    @staticmethod
    def load_records(path: str) -> list[dict]:
        records = []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records

## The `LiveDashboard`

For interactive debugging during a run, `plt.ion()` enables interactive mode. `plt.pause(interval)` updates the display and processes GUI events without blocking the training loop. The trick is to update data on existing `Line2D` objects rather than clearing and replotting — roughly 10× faster and flicker-free:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque


class LiveDashboard:
    """
    Real-time training dashboard using Matplotlib interactive mode.
    Updates every `update_every` steps without blocking the training loop.

    Layout (2×3 grid):
        [Train Loss]  [Eval Loss]      [Learning Rate]
        [Grad Norm]   [Grad/Weight ρ]  [Tokens/sec]
    """

    def __init__(
        self,
        update_every: int = 10,
        window: int = 500,
        smoothing: float = 0.95,
    ):
        self.update_every = update_every
        self.window       = window
        self.smoothing    = smoothing
        self._steps      = deque(maxlen=window)
        self._train_loss = deque(maxlen=window)
        self._train_loss_smooth = deque(maxlen=window)
        self._eval_steps = deque(maxlen=window)
        self._eval_loss  = deque(maxlen=window)
        self._grad_norm  = deque(maxlen=window)
        self._grad_ratio = deque(maxlen=window)
        self._lr         = deque(maxlen=window)
        self._tps        = deque(maxlen=window)
        self._ema        = None
        self._setup_figure()

    def _setup_figure(self):
        plt.ion()
        self.fig = plt.figure(figsize=(15, 8))
        self.fig.suptitle('Training Dashboard', fontsize=13, fontweight='bold')
        gs = gridspec.GridSpec(2, 3, figure=self.fig, hspace=0.4, wspace=0.35)
        self.ax_loss  = self.fig.add_subplot(gs[0, 0])
        self.ax_eval  = self.fig.add_subplot(gs[0, 1])
        self.ax_lr    = self.fig.add_subplot(gs[0, 2])
        self.ax_gnorm = self.fig.add_subplot(gs[1, 0])
        self.ax_ratio = self.fig.add_subplot(gs[1, 1])
        self.ax_tps   = self.fig.add_subplot(gs[1, 2])

        for ax, title, ylabel in [
            (self.ax_loss,  'Train Loss',           'loss'),
            (self.ax_eval,  'Eval Loss',             'loss'),
            (self.ax_lr,    'Learning Rate',         'lr'),
            (self.ax_gnorm, 'Global Grad Norm',      'norm'),
            (self.ax_ratio, 'Grad/Weight Ratio ρ',   'ρ'),
            (self.ax_tps,   'Throughput',            'tok/s'),
        ]:
            ax.set_title(title, fontsize=10)
            ax.set_ylabel(ylabel, fontsize=9)
            ax.set_xlabel('step', fontsize=9)

        # Create empty Line2D objects — we update their data rather than replotting
        (self._line_train,) = self.ax_loss.plot([], [], lw=1.5, color='steelblue',  alpha=0.4)
        (self._line_train_s,) = self.ax_loss.plot([], [], lw=2,   color='steelblue',  label='train')
        (self._line_eval,)  = self.ax_eval.plot([], [], lw=2, color='darkorange', label='eval')
        (self._line_lr,)    = self.ax_lr.plot([], [], lw=2, color='seagreen')
        (self._line_gnorm,) = self.ax_gnorm.semilogy([], [], lw=1.5, color='firebrick')
        (self._line_ratio,) = self.ax_ratio.semilogy([], [], lw=2, color='purple')
        (self._line_tps,)   = self.ax_tps.plot([], [], lw=2, color='teal')

        for ax in [self.ax_loss, self.ax_eval, self.ax_lr, self.ax_gnorm, self.ax_ratio, self.ax_tps]:
            ax.legend(fontsize=8)
        plt.tight_layout()

    def update(
        self,
        step: int,
        train_loss: float,
        learning_rate: float,
        grad_norm: float,
        grad_ratio: float,
        tokens_per_sec: float,
        eval_loss: float = None,
    ):
        # EMA smoothing
        alpha = 1 - self.smoothing
        self._ema = train_loss if self._ema is None else (1 - alpha) * self._ema + alpha * train_loss

        self._steps.append(step)
        self._train_loss.append(train_loss)
        self._train_loss_smooth.append(self._ema)
        self._grad_norm.append(grad_norm)
        self._grad_ratio.append(grad_ratio)
        self._lr.append(learning_rate)
        self._tps.append(tokens_per_sec)

        if eval_loss is not None:
            self._eval_steps.append(step)
            self._eval_loss.append(eval_loss)

        if step % self.update_every != 0:
            return

        steps = list(self._steps)
        self._line_train.set_data(steps, list(self._train_loss))
        self._line_train_s.set_data(steps, list(self._train_loss_smooth))
        self._line_gnorm.set_data(steps, list(self._grad_norm))
        self._line_ratio.set_data(steps, list(self._grad_ratio))
        self._line_lr.set_data(steps, list(self._lr))
        self._line_tps.set_data(steps, list(self._tps))
        if self._eval_steps:
            self._line_eval.set_data(list(self._eval_steps), list(self._eval_loss))

        for ax in [self.ax_loss, self.ax_eval, self.ax_lr, self.ax_gnorm, self.ax_ratio, self.ax_tps]:
            ax.relim()
            ax.autoscale_view()

        self.fig.canvas.draw()
        plt.pause(0.01)  # <1>

1. `plt.pause(0.01)` flushes pending draw commands and yields to the GUI event loop — required to keep the window responsive. The 0.01 second duration is a lower bound; if your training step takes 0.5 seconds the pause is negligible. Increase `update_every` if throughput matters.

## The Sawtooth Chart

A **sawtooth pattern** on any metric is a repeated spike-and-drop. The key is identifying *what* is spiking and *what* is dropping:

| Sawtooth on | Cause | Interpretation |
|---|---|---|
| Train loss | Batch noise or gradient accumulation cycle | Normal — smooth with EMA or SMA |
| Gradient norm | Data ordering artifact, LR phase transition, or bad checkpoint load | Investigate if spikes are growing |
| GPU memory | Memory leak — `.item()` every stored loss, `.cpu()` every stored tensor | Bug — fix immediately |
| Grad/weight ratio | Optimizer state reset — forgot to save/load optimizer state with checkpoint | Resume checkpoints correctly |

: Common sawtooth patterns {tbl-colwidths="[20,40,40]"}

For loss smoothing, use EMA during training and SMA for post-hoc analysis:

In [ ]:
def ema(values: list[float], alpha: float = 0.05) -> list[float]:
    """alpha = 1 - smoothing. Lower alpha = smoother."""
    smoothed, s = [], values[0]
    for v in values:
        s = (1 - alpha) * s + alpha * v
        smoothed.append(s)
    return smoothed

# For post-hoc analysis:
import pandas as pd
# df['loss_sma50'] = df['train_loss'].rolling(window=50, min_periods=1).mean()

## The Eight Training Pathologies

We collect all common failure modes, each with a minimal reproducible example. The `run_experiment` helper runs the training loop, captures diagnostics, and returns them for plotting:

In [ ]:
def run_experiment(model, optimizer, steps=300, label='experiment', clip=1.0):
    """
    Minimal training loop that records everything.
    Returns (train_losses, eval_losses, grad_norms, ratios).
    """
    train_losses, eval_losses, grad_norms, ratios = [], [], [], []
    model.train()
    for step in range(steps):
        x, y = get_batch()
        _, loss = model(x, y)
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  [{label}] NaN/Inf loss at step {step} — stopping")
            train_losses.extend([float('nan')] * (steps - step))
            break
        optimizer.zero_grad()
        loss.backward()
        # Compute stats before clipping
        rs = [
            p.grad.norm().item() / (p.norm().item() + 1e-8)
            for p in model.parameters() if p.grad is not None
        ]
        total_sq = sum(p.grad.norm().item() ** 2
                       for p in model.parameters() if p.grad is not None)
        gnorm = total_sq ** 0.5
        grad_norms.append(gnorm)
        ratios.append(float(np.mean(rs)) if rs else 0.0)
        if clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        train_losses.append(loss.item())
    return train_losses, eval_losses, grad_norms, ratios


def plot_pathology(results: dict, title: str):
    """Plot loss + grad norm for each experiment in results."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ax_loss, ax_norm = axes
    for label, (tl, _, gn, _) in results.items():
        ax_loss.plot(tl, label=label, alpha=0.8)
        ax_norm.semilogy([g for g in gn if np.isfinite(g)], label=label, alpha=0.8)
    ax_loss.set_title(f'{title} — Loss')
    ax_norm.set_title(f'{title} — Grad Norm')
    for ax in axes:
        ax.set_xlabel('Step')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

### Loss Plateau

[Three distinct causes produce identical-looking loss plateaus.]{.mark} Distinguishing them requires looking at the ratio chart:

- **Cause A — LR too small.** The gradient signal is correct but the step size is too small. The ratio looks healthy but the effective update $\text{lr} \cdot \rho$ is tiny.
- **Cause B — LR decayed too aggressively.** The cosine schedule has reached its minimum while the model still has room to improve.
- **Cause C — Dead neurons.** A large fraction of FFN units output near-zero for all inputs. Visible as high `act_frac_zero`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cause A: LR too small
model_a = GPT(config).to(device)
opt_a   = torch.optim.AdamW(model_a.parameters(), lr=1e-7)
res_a   = run_experiment(model_a, opt_a, steps=300, label='lr=1e-7')

# Cause C: dead neurons (very high LR pushes weights into dead zone)
model_c = GPT(config).to(device)
opt_c   = torch.optim.SGD(model_c.parameters(), lr=10.0)
res_c   = run_experiment(model_c, opt_c, steps=300, label='dead neurons', clip=0)

plot_pathology({'lr=1e-7': res_a, 'dead neurons': res_c}, title='Loss Plateau')

| Cause | Fix |
|---|---|
| LR too small | Increase by 10× and observe whether ratio enters healthy band |
| Schedule decayed | Set `min_lr = 0.1 * max_lr` (Chinchilla convention) |
| Dead neurons | Reduce LR, add gradient clipping, use SwiGLU instead of ReLU |

: Loss plateau fixes {tbl-colwidths="[30,70]"}

### Loss Spike

A loss spike is caused by a parameter update that moves weights too far. [The gradient norm spike at step $t$ is the *cause*; the loss spike at step $t+1$ or $t+2$ is the *effect*.]{.mark} Plot `grad_norm[t]` and `train_loss[t+1]` on the same axis to confirm the causal chain.

In [ ]:
model = GPT(config).to(device)
opt   = torch.optim.AdamW(model.parameters(), lr=3e-4)

train_losses, grad_norms = [], []
for step in range(300):
    x, y = get_batch()
    # At step 150 inject an adversarial batch
    if step == 150:
        rare_token = config.vocab_size - 1
        x = torch.full((8, block), rare_token, dtype=torch.long, device=device)
        y = torch.randint(0, config.vocab_size, (8, block), device=device)
        print(f"  Injecting adversarial batch at step {step}")
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    gnorm = sum(p.grad.norm().item() ** 2 for p in model.parameters()
                if p.grad is not None) ** 0.5
    grad_norms.append(gnorm)
    opt.step()   # no clipping — let the spike happen
    train_losses.append(loss.item())

# Find steps where grad norm spiked and check loss lag
norms_arr = np.array(grad_norms)
for i in range(50, len(norms_arr)):
    mean = norms_arr[max(0, i-50):i].mean()
    if norms_arr[i] > 5 * mean and i + 2 < len(train_losses):
        print(f"  spike at step {i}: gnorm={norms_arr[i]:.2f}  "
              f"loss[+1]={train_losses[i+1]:.4f}  loss[+2]={train_losses[i+2]:.4f}")

### Loss Divergence

[Divergence is a positive feedback loop:]{.mark} $\text{large loss} \to \text{large gradient} \to \text{large update} \to \text{larger loss} \to \ldots$. Diagnose by step 50 — if loss has not started decreasing and gradient norms are growing, kill the run immediately and reduce LR by 10×.

Early divergence detector:

In [ ]:
def check_divergence(step: int, loss: float, initial_loss: float, threshold=3.0) -> bool:
    """Returns True if loss has diverged. Ignore the first 20 steps (warmup)."""
    if step < 20:
        return False
    if loss > threshold * initial_loss:
        print(f"  DIVERGENCE at step {step}: "
              f"loss={loss:.4f} > {threshold}×initial={initial_loss:.4f}")
        return True
    return False

### Vanishing Gradients

Without residual connections, the gradient decomposes into a product of Jacobians across layers, each with spectral norm $\leq 1$ for typical activations. The product shrinks exponentially with depth. For the Transformer with residual connections, this should not happen — the residual path guarantees a gradient highway. If you see vanishing gradients in a Transformer, the usual culprits are post-norm (instead of pre-norm) or missing residual scaling:

In [ ]:
#| code-fold: true
class DeepMLP(nn.Module):
    def __init__(self, depth=12, d=256, use_residual=False):
        super().__init__()
        self.use_residual = use_residual
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(depth)])
        self.head   = nn.Linear(d, 1)

    def forward(self, x):
        for layer in self.layers:
            h = torch.tanh(layer(x))
            x = x + h if self.use_residual else h
        return self.head(x)

def measure_layer_gradients(model, x):
    y = model(x).mean()
    y.backward()
    return [layer.weight.grad.norm().item()
            for layer in model.layers if layer.weight.grad is not None]

x = torch.randn(32, 256)
mlp_no_res   = DeepMLP(depth=12, use_residual=False)
mlp_res      = DeepMLP(depth=12, use_residual=True)
norms_no_res = measure_layer_gradients(mlp_no_res, x.clone())
norms_res    = measure_layer_gradients(mlp_res,    x.clone())

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(norms_no_res, 'o-', color='#F44336', lw=2, label='no residuals')
ax.semilogy(norms_res,    's-', color='#4CAF50', lw=2, label='with residuals')
ax.set_xlabel('Layer index (0 = earliest)')
ax.set_ylabel('Gradient norm (log)')
ax.set_title('Vanishing Gradients: Residual vs No-Residual (12-layer MLP)')
ax.axhline(1e-3, color='gray', linestyle='--', lw=0.8)
ax.legend()
plt.tight_layout()
plt.show()

### Exploding Gradients

Exploding gradients occur when weight matrices have spectral norm $> 1$, amplifying the gradient at each layer. The fix is gradient clipping. To calibrate the clip threshold, run 200 steps without clipping and use the 95th percentile of the gradient norm distribution — this means clipping fires on roughly 5% of steps (the outliers) and has no effect otherwise.

### NaN Propagation

[Once any weight contains NaN, the entire model is dead.]{.mark} Three common sources: (A) `log(0)` when a softmax probability is exactly zero; (B) `0/0` in attention when all logits for a query position are $-\infty$; (C) `inf - inf` when logits overflow `exp`. Guard against all three:

In [ ]:
def safe_backward(loss, optimizer, model, step) -> bool:
    """Backward with NaN detection. Returns True if the step was taken."""
    if torch.isnan(loss) or torch.isinf(loss):
        print(f"  Step {step}: NaN/Inf loss — skipping")
        optimizer.zero_grad()
        return False
    optimizer.zero_grad()
    loss.backward()
    nan_params = [
        name for name, p in model.named_parameters()
        if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
    ]
    if nan_params:
        print(f"  Step {step}: NaN gradient in {nan_params[:3]}... — skipping")
        optimizer.zero_grad()
        return False
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return True


# In attention: use -1e9 mask value, not -inf
# -inf causes nan when all positions are masked; -1e9 produces a valid (very peaked) distribution
def safe_softmax(scores, mask=None):
    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)
    scores = scores - scores.max(dim=-1, keepdim=True).values
    return torch.nn.functional.softmax(scores, dim=-1)

### Dead Neurons

[Dead neurons cannot be revived by gradient descent. Prevention is the only cure.]{.underline} Prevention checklist: (1) use SwiGLU instead of ReLU — SwiGLU has nonzero gradient everywhere; (2) gradient clipping stops the large step that kills neurons; (3) `weight_decay=0.1` in AdamW keeps weights from drifting too far.

### Overfitting

For language models, overfitting is more subtle than for classification — a model that memorizes training text assigns high perplexity to held-out text because it has overfit to specific sequences. The train/eval loss gap is the only signal:

- **More data** — the most effective fix.
- **Weight decay** — `weight_decay=0.1` in AdamW.
- **Early stopping** — save the checkpoint where eval loss is minimum.
- **Chinchilla optimal stopping** — for our 29.9M nano model, $D \approx 20N \approx 600\text{M}$ tokens. TinyShakespeare has ~1M tokens — stop after 1 epoch.

## The `HealthReport`

Everything above, automated. Run every 500 steps and get a prioritized list of what to fix:

In [ ]:
from dataclasses import dataclass as _dc
from enum import Enum


class Severity(Enum):
    OK       = "✓"
    WARNING  = "⚠"
    CRITICAL = "✗"


@_dc
class Finding:
    severity: Severity
    check:    str
    detail:   str
    fix:      str


class HealthReport:
    """
    Runs a battery of diagnostics on the recorded training history
    and produces a prioritized list of findings.

    Usage:
        report = HealthReport()
        for step in range(max_steps):
            ...training step...
            report.record(step, train_loss, grad_norm, grad_ratio, lr)
            if step % 500 == 0 and step > 0:
                report.print_report()
    """

    def __init__(self, window: int = 100):
        self.window       = window
        self._train_loss: list[float] = []
        self._eval_loss:  list[tuple] = []
        self._grad_norm:  list[float] = []
        self._grad_ratio: list[float] = []
        self._lr:         list[float] = []
        self._act_zero:   list[float] = []
        self._step       = 0
        self._initial_loss: float | None = None

    def record(
        self,
        step: int,
        train_loss: float,
        grad_norm: float,
        grad_ratio: float,
        lr: float,
        eval_loss: float = None,
        act_frac_zero: float = None,
    ):
        self._step = step
        self._train_loss.append(train_loss)
        self._grad_norm.append(grad_norm)
        self._grad_ratio.append(grad_ratio)
        self._lr.append(lr)
        if eval_loss is not None:
            self._eval_loss.append((step, eval_loss))
        if act_frac_zero is not None:
            self._act_zero.append(act_frac_zero)
        if self._initial_loss is None and len(self._train_loss) >= 10:
            self._initial_loss = np.mean(self._train_loss[:10])

    def _check_plateau(self):
        tl = self._train_loss[-self.window:]
        if len(tl) < 50:
            return None
        recent   = np.mean(tl[-50:])
        earlier  = np.mean(tl[-100:-50]) if len(tl) >= 100 else tl[0]
        improve  = (earlier - recent) / (abs(earlier) + 1e-8)
        if improve < 0.005:
            mean_ratio = np.mean(self._grad_ratio[-50:])
            lr = self._lr[-1] if self._lr else 0
            if mean_ratio < 1e-4 or lr < 1e-6:
                fix = "LR too small or decayed — increase LR or extend schedule"
            else:
                fix = "Possible dead neurons — check act_frac_zero"
            return Finding(
                Severity.WARNING, 'Loss plateau',
                f'<0.5% improvement over last 50 steps (ratio={mean_ratio:.2e})', fix,
            )
        return None

    def _check_divergence(self):
        if self._initial_loss is None or not self._train_loss:
            return None
        loss = self._train_loss[-1]
        if np.isnan(loss):
            return Finding(
                Severity.CRITICAL, 'NaN loss',
                'Loss is NaN — model is dead',
                'Reload last checkpoint; add NaN guard in training loop',
            )
        if loss > 3 * self._initial_loss and self._step > 50:
            return Finding(
                Severity.CRITICAL, 'Loss divergence',
                f'loss={loss:.4f} > 3× initial={self._initial_loss:.4f}',
                'Kill run; reduce LR by 10×; verify initialization',
            )
        return None

    def _check_gradient_health(self):
        if not self._grad_norm:
            return None
        recent_norms = self._grad_norm[-50:]
        if len(recent_norms) < 10:
            return None
        mean_norm = np.mean(recent_norms)
        if mean_norm > 10.0:
            return Finding(
                Severity.WARNING, 'High gradient norms',
                f'mean grad norm over last 50 steps = {mean_norm:.2f}',
                'Check clipping; reduce LR',
            )
        mean_ratio = np.mean(self._grad_ratio[-50:]) if self._grad_ratio else 0.0
        if mean_ratio < 1e-5:
            return Finding(
                Severity.WARNING, 'Possible vanishing gradients',
                f'mean ρ={mean_ratio:.2e} < 1e-5',
                'Check residual scaling; verify pre-norm is active',
            )
        return None

    def _check_overfitting(self):
        if len(self._eval_loss) < 3:
            return None
        steps, evals = zip(*self._eval_loss[-5:])
        if evals[-1] > evals[0] * 1.05:
            gap = evals[-1] - np.mean(self._train_loss[-50:])
            return Finding(
                Severity.WARNING, 'Overfitting',
                f'eval loss rising ({evals[0]:.4f} → {evals[-1]:.4f}); train/eval gap={gap:.4f}',
                'More data; increase weight_decay; early stopping',
            )
        return None

    def _check_dead_neurons(self):
        if not self._act_zero:
            return None
        frac = np.mean(self._act_zero[-20:])
        if frac > 0.5:
            return Finding(
                Severity.WARNING, 'Dead neurons',
                f'mean act_frac_zero={frac:.2f} > 0.5',
                'Reduce LR; add clipping; switch to SwiGLU',
            )
        return None

    def print_report(self):
        checks = [
            self._check_divergence(),
            self._check_gradient_health(),
            self._check_plateau(),
            self._check_dead_neurons(),
            self._check_overfitting(),
        ]
        findings = [c for c in checks if c is not None]
        bar = '═' * 60
        print(f"\n{bar}")
        print(f"  Health Report — step {self._step}")
        print(f"{bar}\n")
        if not findings:
            print(f"  {Severity.OK.value}  All checks passed")
            print(f"     step={self._step}  loss={self._train_loss[-1]:.4f}"
                  f"  ρ={np.mean(self._grad_ratio[-10:]) if self._grad_ratio else 0:.2e}")
        else:
            for f in sorted(findings, key=lambda x: ['CRITICAL', 'WARNING', 'OK'].index(x.severity.name)):
                print(f"  {f.severity.value}  {f.check}")
                print(f"     {f.detail}")
                print(f"     Fix: {f.fix}\n")
        print(f"{bar}\n")

Usage in the training loop:

In [ ]:
report = HealthReport()

for step in range(2000):
    x, y   = get_batch()
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).item()
    optimizer.step()

    ratios = [
        p.grad.norm().item() / (p.norm().item() + 1e-8)
        for p in model.parameters() if p.grad is not None
    ]
    mean_ratio = float(np.mean(ratios)) if ratios else 0.0

    report.record(
        step=step,
        train_loss=loss.item(),
        grad_norm=gnorm,
        grad_ratio=mean_ratio,
        lr=optimizer.param_groups[0]['lr'],
    )

    if step % 500 == 0 and step > 0:
        report.print_report()

## Summary

| Concept | Key detail |
|---|---|
| `register_forward_hook` | Fires after `module.forward()`. Always `.detach()` the output. |
| `register_full_backward_hook` | `grad_output[0]` = gradient flowing into this module. |
| `register_hook` on parameter | Fires after gradient accumulation, before optimizer step. |
| Remove hooks | Store handles, call `.remove()`. Forgotten hooks leak memory. |
| Gradient-to-weight ratio $\rho$ | $\|\nabla W\| / \|W\|$. Healthy range: $[10^{-3}, 10^{-2}]$. |
| Lever 1 — Init | Too small: $\|W\|$ tiny, $\rho$ inflated but weights don't move. Too large: saturation. |
| Lever 2 — Norm | Without pre-norm: activation scale compounds with depth; ratio degrades by layer. |
| Lever 3 — LR | LR does not appear in $\rho$ — it appears in the effective update ratio $\text{lr} \cdot \rho$. |
| Lever 4 — Clipping | Pre-clip norm is the key signal. Consistently at threshold → reduce LR or raise clip. |
| Lever 5 — Residual scaling | $1/\sqrt{2L}$ on output projections. Without it, ratio decays monotonically with depth. |
| Dead neurons | `act_frac_zero > 0.5` → most capacity lost. Not recoverable via SGD. |
| `TrainingLogger` | JSONL append-only file. Crash-safe. Visualization tools come and go. |
| `LiveDashboard` | `plt.ion()` + `set_data()` (not `ax.clear()`). ~10× faster, no flicker. |
| Sawtooth on ratio | Optimizer state not saved/loaded with checkpoint. |
| `HealthReport` | Runs after each epoch. Detects all 8 pathologies. Prioritizes by severity. |

: {tbl-colwidths="[30,70]"}

## Exercises

**1.** Modify `GradientMonitor` to also record the histogram of gradient values (as a `torch.Tensor` of bin counts) for each layer. Plot the histogram at steps 0, 50, 100, 150, and 200 for the first attention output projection. Observe how the distribution changes as training progresses.

**2.** Reproduce the dead neuron experiment deliberately: train with `lr=1e-1` for 50 steps on a 2-layer FFN with ReLU. Plot `act_frac_zero` over time. Then reduce LR to `1e-3` and continue — confirm the dead neurons do not recover.

**3.** Add a `trigger_alert` argument to `GradientMonitor.record()` that calls a user-provided callback whenever any layer's ratio falls outside `[lo, hi]`. Use this to automatically checkpoint the model when a ratio spike is detected.

**4.** Implement a `compare_runs` function that takes a list of JSONL paths and run names, loads all of them into DataFrames, and produces a single figure with all train loss curves (SMA-smoothed, window=50) overlaid on the same axes. Useful for hyperparameter ablations.

**5.** Add a seventh panel to `LiveDashboard` showing GPU memory over time. Handle the case where CUDA is unavailable gracefully.

**6.** Implement `calibrate_clip_threshold(model, dataloader, n_steps=200)` that runs training without clipping for `n_steps` and returns the 95th percentile of pre-clip gradient norms as the recommended clip value.

■